<a href="https://colab.research.google.com/github/cristianmejia00/clustering/blob/main/Topic_Models_using_BERTopic_TOPIC_MODEL_20241101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic Modeling with BERTopic

This notebook runs BERTopic topic modeling on a prepared bibliometric corpus.

**Pipeline position:**
1. Dataset preparation (R pipeline)
2. Embedding generation (EMBEDS notebook or `pipelines/dataset/build_embeddings.py`)
3. Topic modeling (this notebook)
4. Topic diagnostics and reporting

**What this notebook expects:**
- A valid analysis profile (for example `a01`) with its settings JSON
- Precomputed embedding artifacts from the selected embedding profile (for example `e01`)
- Dataset artifacts generated by the dataset pipeline

**What this notebook does:**
1. Loads analysis settings
2. Loads dataset corpus and embeddings
3. Runs UMAP + clustering + BERTopic fitting
4. Produces topic tables and visual diagnostics

**Notes:**
- This notebook is intended to consume artifacts rather than recompute embeddings.
- For consistency, keep analysis and embedding profile names aligned with `config_analysis.yml` and `config_dataset.yml`.

## Requirements and Setup

### Imports and package initialization

In [ ]:
import pandas as pd
import os
import json
import numpy as np
from datetime import date
from itertools import compress
from bertopic import BERTopic
from umap import UMAP
from sklearn.cluster import KMeans

## Inputs and Options

This section defines the runtime paths and profile identifiers used by the analysis settings file.

**Recommended path precedence:**
1. `BIBLIOMETRICS_DRIVE` environment variable
2. `.env` value for `BIBLIOMETRICS_DRIVE`
3. Local fallback path in the notebook

**Expected inputs:**
- Project folder name (for example `Q339_igem`)
- Analysis profile ID (for example `a01_tm__f01_e01__hdbs`)
- Analysis settings JSON filename
- Existing embeddings and corpus artifacts for the profile referenced in analysis settings

If artifacts are missing, run the dataset pipeline and embeddings pipeline first.

In [3]:
# The bibliometrics folder
# Colab
ROOT_FOLDER_PATH = "drive/MyDrive/Bibliometrics_Drive"

# Mac
ROOT_FOLDER_PATH = "/Users/cristian/Library/CloudStorage/GoogleDrive-cristianmejia00@gmail.com/My Drive/Bibliometrics_Drive"

# Change to the name of the folder where the dataset is uploaded inside the above folder
project_folder = 'Q339_igem'

# Analysis ID
analysis_id = 'a01_tm__f01_e01__hdbs'

# Filtered label
settings_directive = "settings_analysis_directive_2025-08-04-16-40.json"

In [4]:
# Read settings
with open(f'{ROOT_FOLDER_PATH}/{project_folder}/{analysis_id}/{settings_directive}', 'r') as file:
    settings = json.load(file)

In [5]:
# Input dataset
dataset_file_path = f"{ROOT_FOLDER_PATH}/{settings['metadata']['project_folder']}/{settings['metadata']['filtered_folder']}/dataset_raw_cleaned.csv"

In [6]:
# Function to save files
def save_as_csv(df, save_name_without_extension, with_index):
    "usage: `save_as_csv(dataframe, 'filename')`"
    df.to_csv(f"{ROOT_FOLDER_PATH}/{save_name_without_extension}.csv", index=with_index)
    print("===\nSaved: ", f"{ROOT_FOLDER_PATH}/{save_name_without_extension}.csv")

In [9]:
# Open the data file
df = pd.read_csv(f"{dataset_file_path}", encoding='latin-1', usecols=['X_N', 'uuid', 'UT', 'PY', 'TI', 'AB'])
print(df.shape)
df.head()

(4548, 6)


,X_N,uuid,UT,PY,TI,AB
0,1,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,173,2009,Ethanol? Whey not!,Cheese whey is classified as a special waste f...
1,2,f7173d50-7003-491e-a6ff-527fc1055e00,174,2009,Bac-man: sequestering cadmium into Bacillus sp...,Cadmium contamination can be a serious problem...
2,3,8169f0c0-96ec-4ee5-9458-bb03b9f1ef28,175,2009,Bacterial Relay Race,"In our project, we aim at creating a cell-to-c..."
3,4,f0c10215-5236-419b-8112-635791483c0a,176,2009,E. coli Automatic Directed Evolution Machine: ...,Evolution is powerful enough to create everyth...
4,5,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,177,2009,BacInVader â a new system for cancer genetic...,The main aim of our project is to design a mod...




---



## PART 2: Topic Model

In [10]:
# bibliometrics_folder
# project_folder
# project_name_suffix
# ROOT_FOLDER_PATH = f"drive/MyDrive/{bibliometrics_folder}"

#############################################################
# Embeddings folder
embeddings_folder_name = settings['tmo']['embeds_folder']

# Which column has the year of the documents?
my_year = settings['tmo']['year_column']

# Number of topics. Select the number of topics to extract.
# Choose 0, for automatic detection.
n_topics = 71#settings['tmo']['n_topics']

# Minimum number of documents per topic
min_topic_size = settings['tmo']['min_topic_size']

# Threshold for others
# This is the threshold for the topics that will be considered as "others"
# Topics with a cumulative percentage of documents below this threshold will be grouped into an "others" topic.
# For example, if set to 0.9, topics that together account for 90% of the documents will be kept, and the rest will be grouped into "others".
others_threshold = 0.99#settings['tmo']['others_threshold']

In [ ]:
# Load precomputed embeddings artifacts from dataset pipeline
embeds_base = f"{ROOT_FOLDER_PATH}/{project_folder}/{settings['metadata']['filtered_folder']}/{embeddings_folder_name}"

embeddings_array = np.load(f"{embeds_base}/embeddings.npy")
with open(f"{embeds_base}/embeddings_ids.json", "r") as f:
    embeddings_ids = json.load(f)

corpus = pd.read_csv(f"{embeds_base}/corpus.csv").reset_index(drop=True)
documents = corpus.text.to_list()

embeddings = {
    "embeddings": embeddings_array,
    "embeddings_ids": embeddings_ids,
}
print(f"Embeddings shape: {embeddings['embeddings'].shape}")

Embeddings shape: (4548, 384)


In [ ]:
# verify that the number of embeddings matches the number of documents
assert(len(embeddings['embeddings']) == len(documents))

# verify that embedding and documents are in the same order as the original dataframe
id_col = 'uuid' if 'uuid' in corpus.columns else ('UT' if 'UT' in corpus.columns else None)
assert(id_col is not None), "Neither 'uuid' nor 'UT' column found in corpus.csv"
assert(corpus[id_col].astype(str).tolist() == [str(x) for x in embeddings['embeddings_ids']])

---

# Topic Modeling with BERTopic

In [13]:
from hdbscan.hdbscan_ import HDBSCAN
# Execute the topic model.
# I suggest changing the values marked with #<---
# The others are the default values and they'll work fine in most cases.
# This will take several minutes to finish.

# Initiate UMAP
umap_model = UMAP(n_neighbors=15,
                  n_components=5,
                  min_dist=0.0,
                  metric='cosine',
                  random_state=100)

if n_topics == 0:
  # Initiate topic model with HDBScan (Automatic topic selection)
  topic_model_params = HDBSCAN(min_cluster_size=min_topic_size,
                               metric='euclidean',
                               cluster_selection_method='eom',
                               prediction_data=True)
else:
  # Initiate topic model with K-means (Manual topic selection)
  topic_model_params = KMeans(n_clusters = n_topics)

# Initiate BERTopic
topic_model = BERTopic(umap_model = umap_model,
                       hdbscan_model = topic_model_params,
                       min_topic_size=min_topic_size,
                       #nr_topics=15,          #<--- Footnote 1
                       n_gram_range=(1,3),
                       language='english',
                       calculate_probabilities=True,
                       verbose=True)



# Footnote 1: This controls the number of topics we want AFTER clustering.
# Add a hashtag at the beggining to use the number of topics returned by the topic model.
# When using HDBScan nr_topics will be obtained after orphans removal, and there is no warranty that `nr_topics < HDBScan topics`.
# thus, with HDBScan `nr_topics` means N topics OR LESS.
# When using KMeans nr_topics can be used to further reduce the number of topics.
# We use the topics as returned by the topic model. So we do not need to activate it here.

In [ ]:
# Compute topic model
topics, probabilities = topic_model.fit_transform(documents, embeddings['embeddings'])

2026-03-10 14:29:05,568 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-10 14:29:18,107 - BERTopic - Dimensionality - Completed ✓
2026-03-10 14:29:18,108 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-10 14:29:18,173 - BERTopic - Cluster - Completed ✓
2026-03-10 14:29:18,175 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-10 14:29:20,806 - BERTopic - Representation - Completed ✓


In [15]:
# Get the list of topics
# Topic = the topic number. From the largest topic.
#         "-1" is the generic topic. Genericr keywords are aggegrated here.
# Count = Documents assigned to this topic
# Name = Top 4 words of the topic based on probability
# Representation = The list of words representing this topic
# Representative_Docs = Documents assigned to this topic
tm_summary = topic_model.get_topic_info()
tm_summary

,Topic,Count,Name,Representation,Representative_Docs
0,0,164,0_cancer_cells_the_of,"[cancer, cells, the, of, and, to, in, therapy,...",[The Boomerang system engineering logic gate ...
1,1,129,1_of_the_and_to,"[of, the, and, to, in, we, for, synthetic, evo...",[SciPhi Enabling orthogonal replication and p...
2,2,129,2_the_to_of_and,"[the, to, of, and, plant, in, by, is, that, for]",[Project FEDDS Developing an Integrated Fluore...
3,3,121,3_the_detection_of_diagnostic,"[the, detection, of, diagnostic, and, to, in, ...",[Argonaute Mediated Viral Point of Care Diagno...
4,4,121,4_metal_heavy_cadmium_metals,"[metal, heavy, cadmium, metals, heavy metal, a...",[Nanocrystal Ecoli Flocculation Union Heavy me...
...,...,...,...,...,...
66,66,23,66_vocs_volatile_olfactory_the,"[vocs, volatile, olfactory, the, breath, volat...",[Vigilantly Optimizing Cancer Detection Using ...
67,67,23,67_biofilm_biofilms_the_of,"[biofilm, biofilms, the, of, to, and, corrosio...",[Engineering Ecoli to detect and destroy biofi...
68,68,22,68_magnetic_magnetosome_magnetotactic_to,"[magnetic, magnetosome, magnetotactic, to, mag...",[Cadmium recovery by a recombinant Magnetospir...
69,69,19,69_dental_caries_dental caries_mutans,"[dental, caries, dental caries, mutans, oral, ...",[ToothkeeperA novel scheme for early diagnosis...


In [ ]:
# Save the topic model assets
tm_folder_path = f'{ROOT_FOLDER_PATH}/{project_folder}/{settings["metadata"]["analysis_id"]}'

os.makedirs(tm_folder_path, exist_ok=True)

tm_summary.to_csv(f'{tm_folder_path}/topic_model_info.csv', index=False)

In [17]:
# Number of topics found
found_topics = max(tm_summary.Topic) + 1
found_topics

71

In [18]:
# Confirm all documents are assigned
sum(tm_summary.Count) == len(corpus)

True

In [19]:
# Get the top 10 documents for a topic
topic_model.get_representative_docs(0)

['The Boomerang system  engineering logic gate genetic device for detection and treatment of cancer Despite recent treatment advancements cancer is still a major cause of mortality worldwide One of the fundamental problems preventing the development of effective therapy is the difficulty to target cancer cells exclusively In Boomerang were engineering a genetic device based on a simple concept of AND logic gate the activation of our CRISPRCasbased system is dependent on the existence of two cancerspecific promoters that control the expression of Cas and gRNA and the combination of these two will occur only in cancer cells CRISPRCas system allows several applications of Boomerang  disruption of genes essential for cancer survival and  activation of suicide genes or color proteins for cancer cell detection eg for complete surgical removal Our system can be potentially designed according to unique characteristics of a patients tumor paving the way to personalized medicine We hope that our

In [20]:
# Print the parameters used. (For reporting)
topic_model.get_params()

{'calculate_probabilities': True,
 'ctfidf_model': ClassTfidfTransformer(),
 'embedding_model': None,
 'hdbscan_model': KMeans(n_clusters=71),
 'language': 'english',
 'low_memory': False,
 'min_topic_size': 5,
 'n_gram_range': (1, 3),
 'nr_topics': None,
 'representation_model': None,
 'seed_topic_list': None,
 'top_n_words': 10,
 'umap_model': UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=5, n_jobs=1, random_state=100, tqdm_kwds={'bar_format': '{desc}: {percentage:3.0f}%| {bar} {n_fmt}/{total_fmt} [{elapsed}]', 'desc': 'Epochs completed', 'disable': True}),
 'vectorizer_model': CountVectorizer(ngram_range=(1, 3)),
 'verbose': True,
 'zeroshot_min_similarity': 0.7,
 'zeroshot_topic_list': None}

In [21]:
tm_params = dict(topic_model.get_params())
for key, value in tm_params.items():
    tm_params[key]=  str(value)
with open(f'{tm_folder_path}/topic_model_params.json', 'w') as f:
    json.dump(tm_params, f, ensure_ascii=False, indent=4)
    print('Done')

Done


---
# Centroids

In [22]:
import numpy as np
from sklearn.metrics.pairwise import cosine_distances

def calculate_centroid_scores(embeddings, topics, distance_metric='euclidean'):
    """
    Vectorized calculation of distance from topic centroids.
    Returns normalized scores (0-1) where 0 is the centroid and 1 is far.
    """
    embeddings = np.array(embeddings)
    topics = np.array(topics)
    unique_topics = np.unique(topics)
    
    scores = np.zeros(len(embeddings))
    
    # Calculate centroids and distances
    for topic in unique_topics:
        if topic == -1: continue # Skip outliers for centroid calculation
        
        mask = topics == topic
        topic_embeds = embeddings[mask]
        centroid = np.mean(topic_embeds, axis=0)
        
        if distance_metric == 'euclidean':
            dists = np.linalg.norm(topic_embeds - centroid, axis=1)
        else:
            dists = cosine_distances([centroid], topic_embeds)[0]
            
        # Normalize 0-1 within topic
        if len(dists) > 1 and dists.max() > dists.min():
            scores[mask] = (dists - dists.min()) / (dists.max() - dists.min())
        else:
            scores[mask] = 0 # Single document or identical distances
            
    return scores

In [23]:
# Usage with normalization:
scores = calculate_centroid_scores(embeddings['embeddings'], topics)

---
# Coordinates

In [24]:
dataset_clustering_results = topic_model.get_document_info(documents, df = corpus)
dataset_clustering_results.head()

,text,UT,uuid,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
0,Ethanol Whey not Cheese whey is classified as ...,173,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,Ethanol Whey not Cheese whey is classified as ...,18,18_the_vitamin_and_of,"[the, vitamin, and, of, to, in, production, mi...",[Dlicious Engineering Butyrateproducing Bacter...,the - vitamin - and - of - to - in - productio...,False
1,Bacman sequestering cadmium into Bacillus spor...,174,f7173d50-7003-491e-a6ff-527fc1055e00,Bacman sequestering cadmium into Bacillus spor...,4,4_metal_heavy_cadmium_metals,"[metal, heavy, cadmium, metals, heavy metal, a...",[Nanocrystal Ecoli Flocculation Union Heavy me...,metal - heavy - cadmium - metals - heavy metal...,False
2,Bacterial Relay Race In our project we aim at ...,175,8169f0c0-96ec-4ee5-9458-bb03b9f1ef28,Bacterial Relay Race In our project we aim at ...,14,14_of_the_to_in,"[of, the, to, in, and, system, we, is, circuit...",[PERspectives Invitro development of PERceptro...,of - the - to - in - and - system - we - is - ...,False
3,E coli Automatic Directed Evolution Machine a ...,176,f0c10215-5236-419b-8112-635791483c0a,E coli Automatic Directed Evolution Machine a ...,1,1_of_the_and_to,"[of, the, and, to, in, we, for, synthetic, evo...",[SciPhi Enabling orthogonal replication and p...,of - the - and - to - in - we - for - syntheti...,False
4,BacInVader a new system for cancer genetic th...,177,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,BacInVader a new system for cancer genetic th...,7,7_cancer_tumor_cells_the,"[cancer, tumor, cells, the, and, to, of, in, t...",[Targeted Treatment of Colorectal Cancer with ...,cancer - tumor - cells - the - and - to - of -...,False


In [25]:
from sklearn.manifold import TSNE

# 6. VISUALIZATION (Switching to t-SNE for the "Map" look)
print("Generating 2D coordinates for visualization...")

# Filter out outliers (-1) to keep the visualization clean
valid_indices = dataset_clustering_results['Topic'] != -1
valid_embeddings = np.array(embeddings['embeddings'])[valid_indices]
df_clean = dataset_clustering_results[valid_indices].reset_index(drop=True)

# --- KEY CHANGE: Use t-SNE instead of UMAP for the layout ---
# metric='cosine' is better for embeddings
# init='pca' is CRITICAL: it gives you the "ball" shape instead of random clusters
# perplexity=30-50: balances local vs global structure
tsne_model = TSNE(
    n_components=2, 
    perplexity=50,        # Higher values = more global structure
    learning_rate='auto',
    init='pca',           # This creates the global "continent" shape
    #n_iter=1000, 
    metric='cosine', 
    random_state=42, 
    n_jobs=-1             # Use all CPU cores
)

coords_2d = tsne_model.fit_transform(valid_embeddings)

# Scale coordinates to fill the frame nicely (optional but helps visualization)
# This creates that "filled frame" look
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-100, 100))
coords_2d = scaler.fit_transform(coords_2d)


Generating 2D coordinates for visualization...


In [26]:
# Add coordinates as columns to df_clean
df_clean['x_coords_tsne'] = coords_2d[:, 0]
df_clean['y_coords_tsne'] = coords_2d[:, 1]

In [27]:
coords_2d

array([[ 33.336258 ,  65.50946  ],
       [-76.32136  , -46.922127 ],
       [  5.4127984, -13.836423 ],
       ...,
       [ 76.35961  , -43.129883 ],
       [-60.93066  ,  31.40921  ],
       [ 50.37821  ,  54.1825   ]], shape=(4548, 2), dtype=float32)

---
# Save

In [28]:
# Save coords. Coords cover only the non-orphans documents.
df_clean[['uuid', 'UT', 'x_coords_tsne', 'y_coords_tsne']].to_csv(f'{tm_folder_path}/document_coords_tsne.csv', index=False)

In [29]:
# Document information. Including the topic assignation
dataset_clustering_results = topic_model.get_document_info(documents, df = corpus, metadata={"Score": scores})

# Check for orphans (Topic == -1), save and remove them
if -1 in dataset_clustering_results['Topic'].values:
    orphans = dataset_clustering_results[dataset_clustering_results['Topic'] == -1]
    orphans.to_csv(f'{tm_folder_path}/orphans.csv', index=False)
    dataset_clustering_results = dataset_clustering_results[dataset_clustering_results['Topic'] != -1]

In [30]:
# Standar format for report analysis
dataset_clustering_results = dataset_clustering_results.reset_index(drop=True)
dataset_clustering_results = dataset_clustering_results.drop(columns=['text'])
dataset_clustering_results['X_E'] = dataset_clustering_results['Score']
dataset_clustering_results['X_C'] = dataset_clustering_results['Topic'] + 1

# Assign 'level0' based on cluster coverage (90%)
cluster_counts = dataset_clustering_results['X_C'].value_counts().sort_values(ascending=False)
total_rows = len(dataset_clustering_results)
cumulative = cluster_counts.cumsum() / total_rows
main_clusters = cluster_counts.index[cumulative <= others_threshold].tolist()
dataset_clustering_results['level0'] = dataset_clustering_results['X_C'].apply(lambda x: x if x in main_clusters else 99999)
dataset_clustering_results.head()


,UT,uuid,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document,Score,X_E,X_C,level0
0,173,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,Ethanol Whey not Cheese whey is classified as ...,18,18_the_vitamin_and_of,"[the, vitamin, and, of, to, in, production, mi...",[Dlicious Engineering Butyrateproducing Bacter...,the - vitamin - and - of - to - in - productio...,False,0.228616,0.228616,19,19
1,174,f7173d50-7003-491e-a6ff-527fc1055e00,Bacman sequestering cadmium into Bacillus spor...,4,4_metal_heavy_cadmium_metals,"[metal, heavy, cadmium, metals, heavy metal, a...",[Nanocrystal Ecoli Flocculation Union Heavy me...,metal - heavy - cadmium - metals - heavy metal...,False,0.534650,0.534650,5,5
2,175,8169f0c0-96ec-4ee5-9458-bb03b9f1ef28,Bacterial Relay Race In our project we aim at ...,14,14_of_the_to_in,"[of, the, to, in, and, system, we, is, circuit...",[PERspectives Invitro development of PERceptro...,of - the - to - in - and - system - we - is - ...,False,0.175166,0.175166,15,15
3,176,f0c10215-5236-419b-8112-635791483c0a,E coli Automatic Directed Evolution Machine a ...,1,1_of_the_and_to,"[of, the, and, to, in, we, for, synthetic, evo...",[SciPhi Enabling orthogonal replication and p...,of - the - and - to - in - we - for - syntheti...,False,0.362345,0.362345,2,2
4,177,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,BacInVader a new system for cancer genetic th...,7,7_cancer_tumor_cells_the,"[cancer, tumor, cells, the, and, to, of, in, t...",[Targeted Treatment of Colorectal Cancer with ...,cancer - tumor - cells - the - and - to - of -...,False,0.296230,0.296230,8,8


In [31]:
# Standar format for report analysis
dataset_clustering_results['cl99'] = [True if x == 99999 else False for x in dataset_clustering_results['level0']]
dataset_clustering_results['cl-99'] = [True if x == 99999 else False for x in dataset_clustering_results['level0']]
dataset_clustering_results.head()

,UT,uuid,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document,Score,X_E,X_C,level0,cl99,cl-99
0,173,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,Ethanol Whey not Cheese whey is classified as ...,18,18_the_vitamin_and_of,"[the, vitamin, and, of, to, in, production, mi...",[Dlicious Engineering Butyrateproducing Bacter...,the - vitamin - and - of - to - in - productio...,False,0.228616,0.228616,19,19,False,False
1,174,f7173d50-7003-491e-a6ff-527fc1055e00,Bacman sequestering cadmium into Bacillus spor...,4,4_metal_heavy_cadmium_metals,"[metal, heavy, cadmium, metals, heavy metal, a...",[Nanocrystal Ecoli Flocculation Union Heavy me...,metal - heavy - cadmium - metals - heavy metal...,False,0.534650,0.534650,5,5,False,False
2,175,8169f0c0-96ec-4ee5-9458-bb03b9f1ef28,Bacterial Relay Race In our project we aim at ...,14,14_of_the_to_in,"[of, the, to, in, and, system, we, is, circuit...",[PERspectives Invitro development of PERceptro...,of - the - to - in - and - system - we - is - ...,False,0.175166,0.175166,15,15,False,False
3,176,f0c10215-5236-419b-8112-635791483c0a,E coli Automatic Directed Evolution Machine a ...,1,1_of_the_and_to,"[of, the, and, to, in, we, for, synthetic, evo...",[SciPhi Enabling orthogonal replication and p...,of - the - and - to - in - we - for - syntheti...,False,0.362345,0.362345,2,2,False,False
4,177,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,BacInVader a new system for cancer genetic th...,7,7_cancer_tumor_cells_the,"[cancer, tumor, cells, the, and, to, of, in, t...",[Targeted Treatment of Colorectal Cancer with ...,cancer - tumor - cells - the - and - to - of -...,False,0.296230,0.296230,8,8,False,False


In [32]:
dataset_clustering_results.level0.value_counts()

level0
1     164
2     129
3     129
4     121
5     121
     ... 
64     26
65     25
66     24
67     23
68     23
Name: count, Length: 69, dtype: int64

In [33]:
# Save the dataframe
dataset_clustering_results.to_csv(f'{tm_folder_path}/dataset_minimal.csv', index=False)

In [34]:
# Save the topic model
topic_model.save(f'{tm_folder_path}/topic_model_object.pck')

2026-03-10 14:29:33,264 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [35]:
df_clean['uuid'].isin(dataset_clustering_results['uuid'])

0       True
1       True
2       True
3       True
4       True
        ... 
4543    True
4544    True
4545    True
4546    True
4547    True
Name: uuid, Length: 4548, dtype: bool